## **Project Title: Real-Time Fraud Detection System with Explainable AI & Live Dashboard**

### Problem Statement:
- Financial fraud causes trillions of dollars in global losses every year, making real-time fraud detection essential for modern financial systems. Traditional rule-based methods often fail to detect evolving fraud patterns, while many machine learning models lack transparency and interpretability.

- This project aims to build an end-to-end Fraud Detection System using machine learning techniques capable of identifying fraudulent transactions accurately despite severe class imbalance in the dataset. The system will integrate Explainable AI (SHAP) to provide clear explanations for predictions, helping analysts and stakeholders understand why transactions are flagged as suspicious.

- The final solution will include data preprocessing, fraud prediction models, performance evaluation, and an interactive dashboard for visualizing fraud insights and model explanations.

### Datset Overview

- The IEEE-CIS Fraud Detection Dataset is a large-scale real-world e-commerce transaction dataset released by Vesta Corporation for fraud detection research and machine learning benchmarking. The objective is to predict whether an online transaction is fraudulent using transaction-level and identity-related features.

- Kaggle Link: https://www.kaggle.com/competitions/ieee-fraud-detection/data


- Dataset Size and Structure:
    - Total Transactions: ~590,540
    - Total Features After Merge: 433–434
    - Fraud Rate: ~3.5%
    - Problem Nature: Highly Imbalanced


### **Data Loading, Merging & Exploratory Analysis**

In [ ]:
import pandas as pd     # Importing the pandas library for data manipulation and analysis
import numpy as np      # Importing the numpy library for numerical operations

#### We first need to load both the datasets into respective dataframes:

In [ ]:
df_transaction = pd.read_csv("C:\\Users\\KIIT0001\\OneDrive\\Documents\\VSC\\XYlofy AI\\FraudDetection_AyushPrasad\\data\\train_transaction.csv")
# Loading and reading the transaction dataset into df_transaction

df_identity = pd.read_csv("C:\\Users\\KIIT0001\\OneDrive\\Documents\\VSC\\XYlofy AI\\FraudDetection_AyushPrasad\\data\\train_identity.csv")
# Loading and reading the identity dataset into df_identity

- train_transaction.csv was read and stored in df_transaction.
- train_identity.csv was read and stored in df_identity.

In [ ]:
print(df_transaction.columns)
print(df_identity.columns)          # Displaying the column names of both datasets to understand their structure

- Next, we need to merge both the datasets on 'TransactionID' since it is the common column and not every transaction has identity information.

In [ ]:
df = pd.merge(df_transaction, df_identity, on = 'TransactionID', how = 'left')      # Merging both the datasets on the 'TransactionID' column using a left join to create a combined dataset

- A left join was used while merging **train_transaction.csv** and **train_identity.csv** because the **transaction dataset is the primary dataset** and contains all transactions, while the identity dataset only contains identity information for some transactions.

- This is important because the **company wants information on all the transactions** and not just the ones with identity information.

#### Displaying the structure and sample records of the datset:

In [ ]:
print("Dataset Shape:")
print(df.shape)      # Displaying the shape of the merged dataset

In [ ]:
print("Dataset info:")
print(df.dtypes)                    # Displaying the data types of each column in the merged dataset
print("\nValue Counts:")
print(df.dtypes.value_counts())     # Displaying the count of each data type in the merged dataset

In [ ]:
from IPython.display import display             # Importing the display function from IPython to display the DataFrame in better and a more readable format


print("First 10 rows of the merged dataset:")
display(df.head(10))                            # Displaying the first 10 rows of the merged dataset

- 'isFraud' is the target column where:
    - 0 -> Normal Transaction
    - 1 -> Fraudulent Transaction

#### Quantifying the class imbalance:

In [ ]:
fraud_counts = df['isFraud'].value_counts()
print(fraud_counts)                             # Displaying the count of fraudulent and normal transactions in the dataset

In [ ]:
fraud_perc = df['isFraud'].value_counts(normalize=True) * 100
print(fraud_perc)                                                   # Displaying the percentage of fraudulent and normal transactions in the dataset

#### Raw count values are normalized into proportions of each unique value to calculate the percentage distribution
- Fraudulent Transactions: 96.501%
- Normal Transactions: 3.499%

- This difference in class difference represents there exists massive **Class Imbalance.**

#### Visualizing Class Imbalance:

In [ ]:
import matplotlib.pyplot as plt             # Importing the matplotlib library for data visualization
import seaborn as sns                       # Importing the seaborn library for statistical data visualization

plt.figure(figsize=(9, 6))

sns.countplot(x='isFraud', data=df)         # Creating a count plot to visualize the class imbalance in the dataset

plt.title("Class Imbalance in Dataset")
plt.xlabel("isFraud")
plt.ylabel("Count")
plt.show()



- The chart clearly displays the class imbalance where Normal Transactions dominate the dataset.
- Fraudulent transactions represents only a small percentage (3.499%).

- This imbalance can cause machine learnign models to become biased to predicting the majority class, so imbalance handing(SMOTE) will be used later.

#### Identiying Missing Values:

In [ ]:
missing_values = df.isnull().sum()

print(missing_values)                      # Displaying the count of missing values in each column of the merged dataset

- We need to identify missing values percentage to decide what columns to drop

In [ ]:
missing_value_percent = (missing_values / len(df)) * 100
print(missing_value_percent)                                    # Displaying the percentage of missing values in each column of

- Here the number of missing values were used to calculate the percentage of missing values for each column.
- This will help identify what columns are required to drop.

#### Creating a separate DataFrame for missing values:

In [ ]:
df_missing = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage": missing_value_percent     # Defining a new DataFrame to display the count and percentage of missing values in each column of the merged dataset
})

display(df_missing)                         # Displaying the new missing values DataFrame

- Now we need to only **identify the columns with more than 50% missing values.**

In [ ]:
columns_to_drop = df_missing[
    df_missing["Percentage"] > 50                             # Identifying columns that have more than 50% missing values 
].index

display(columns_to_drop)                                      # Displaying the columns that have more than 50% missing values

print("Number of Columns to Drop:", len(columns_to_drop))     # Displaying the number of columns that have more than 50% missing values

- From this operation we found that there are **214 columns** present in the dataset with more than 50% missing values.
- These columns will be dropped as they **may introduce noise and reduce model quality.**
- Removing these columns will also help in **improving efficiency.**

#### Plotting Distribution of TransactionAmt for fraud vs non-fraud:

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data = df,
    x = "TransactionAmt",
    hue = "isFraud",
    log_scale = True,
    bins = 50
)


plt.title("Distribution of Transaction Amounts for fraud vs non-fraud Transactions")
plt.xlabel("Transaction Amount (log scale)")
plt.ylabel("Count")

plt.show()

- From the chart above, fraudulent transactions appear to have different spending patterns compared to legitimate transactions
- This suggests that transaction amount may be an important predictive feature.

#### Creating a Correlation Heatmap:

- To create this correlation heatmap, we first need to select only the numerical columns (top 20).

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns      # Selecting only the numerical columns for correlation analysis

target_corr = df[numerical_cols].corr()['isFraud']                  # Calculating the correlation of each numerical feature with the target variable 'isFraud'

- Out of all the numerical features, we need only the top (highest correlated values) features.

In [ ]:
top_features = target_corr.abs().sort_values(ascending=False).head(20).index    # Identifying the top 20 features.

print(top_features)

In [ ]:
corr_matrix = df[top_features].corr()       # Calculating the correlation matrix for the top 20 features

In [ ]:
plt.figure(figsize=(14, 10))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5
)                       # Creating a heatmap to visualize the correlation of the top 20 numerical features with the target variable 'isFraud'

plt.title("Top 20 Numerical Features correlation with 'isFraud'")
plt.xticks(rotation=45)

plt.show()

- **V257, V246, V244, V242 have the highest correlation with 'isFraud' (~0.36–0.38).**


- Many features are also strongly correlated with each other.
    - Example:
        - V244 - V242 = 0.97
        - V156 - V149 = 0.98
        - V201 - V200 = 0.94

- Significant correlations among different features were also observed, indicating potential multicollinearity.

### **Preprocessing, Imbalance Handling & Feature Engineering**

- Since we already identified the columns with more than 50% missing values, we will now drop those columns.
- We will impute remaining values using median and mode.

In [ ]:
df = df.drop(columns=columns_to_drop)       # Drop columns with missing values more than 50%

print("New Dataset Shape: ",df.shape)

#### To impute the remaining values, we first need to separate numerical and categorical columns:

- Identify the columns types:

In [ ]:
numerical_cols = df.select_dtypes(
    include=['int64', 'float64']
).columns

categorical_cols = df.select_dtypes(
    include = ['object']
).columns

print("Numerical Columns: ", len(numerical_cols))
print("Categorical Columns: ", len(categorical_cols))

#### Missing Value Imputation
- Numerical - Median
- Categorical - Mode

    - For numerical values, using median is preferreably better than using mode as median is robust against outliers.

In [ ]:
df[numerical_cols] = df[numerical_cols].fillna(
    df[numerical_cols].median()
)                                                   # Impute numerical columns using median

In [ ]:
df[categorical_cols] = df[categorical_cols].apply(
    lambda col: col.fillna(col.mode()[0]) if not col.mode().empty else col      # Impute categorical data using mode
)                                                   

- For median imputation, Pandas can compute all medians directy in one operation but for mode imputation, each columns may have a different value, so the operation is applied column-wise.

In [ ]:
print(df.isnull().sum().sum())      # Verify if there are any missing values left

- This indicates that any missing values have been dropped or imputated.

#### Encoding categorical data:
- Machine learning models cannot process raw text directly, so for that we convert categories into numbers.
- To achieve this we will use **Label-Encoder**

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoders = {}

for col in categorical_cols:
    
    lab_enc = LabelEncoder()                                # Create a Label-Encoder object
    
    df[col] = lab_enc.fit_transform(df[col].astype(str))    # Convert column values to string type then fit and transform categorical values into numerical labels
    
    label_encoders[col] = lab_enc                           # Store the fitted encoder in the dictionary

- Label Encoding was selected for high-cardinality categorical features because:

    - The dataset contains many categorical columns with a large number of unique values.
    - One-Hot Encoding would significantly increase dimensionality and memory usage.
    - Tree-based models such as LightGBM and XGBoost can efficiently handle label-encoded categorical values.
    - Label Encoding improves computational efficiency while preserving category distinctions.

#### Feature Engineering:

Feature 1:
- **AmtToMeanRatio = TransactionAmt / mean(TransactionAmt)**

In [ ]:
df["AmtToMeanRatio"] = (
    df["TransactionAmt"] / df["TransactionAmt"].mean()
)                                                           # Creating a new feature 'AmtToMeanRatio' 

- We created a new feature using 'TransactionAmt' and mean of the values of 'TransactionAmt'.

Feature 2:
- **HourOfDay = extracted from TransactionDT**

In [ ]:
df["HourOfDay"] = (
    (df["TransactionDT"] / 3600) % 24
).astype(int)

- A new feature, "HourOfDay" is created using 'TransactionDT'. It is converted into hours as it is given as seconds in the dataset.
- Used **'% 24'** to keep only the hour within the day and converted it into integer type, as it removes decimals and converts to integer hours.

Feature 3:
- **DeviceRisk = binary flag based on DeviceType and DeviceInfo**


- The **DeviceType** and **DeviceInfo** columns were previously removed due to high missing values during preprocessing.  
To engineer this feature, these columns are temporarily restored from the identity dataset using **TransactionID** as the primary key.

In [ ]:
device_cols = df_identity[
    ["TransactionID", "DeviceType", "DeviceInfo"]       # Restore Device based columns and store it in device_cols
]

In [ ]:
df = df.merge(
    device_cols,
    on="TransactionID",
    how="left"
)                                   # Merge the device_cols with the current dataset using left join on 'TransactionID'

The device-related columns are merged back into the working dataframe using a left join on 'TransactionID'.

This ensures that all existing transactions are preserved.

In [ ]:
df["DeviceType"] = df["DeviceType"].fillna("Unknown")   # Replace missing values with "Unknown" for both 'DeviceType' and 'DeviceInfo'
df["DeviceInfo"] = df["DeviceInfo"].fillna("Unknown")

Missing values in 'DeviceType' and 'DeviceInfo' are replaced with 'Unknown'.

In [ ]:
device_fraud_rate = df.groupby(
    ["DeviceType", "DeviceInfo"]
)["isFraud"].mean()                     # Calculate the Fraud-Probability for 'DeviceType' and 'DeviceInfo'

Fraud probability is calculated for each unique combination of:
- 'DeviceType'
- 'DeviceInfo'

In [ ]:
threshold = device_fraud_rate.quantile(0.90)        # Define only the top 10% of the most risky device combinations as high risk by calculating the 90th percentile of the fraud rates for all device combinations


df["DeviceRisk"] = df.apply(
    lambda row: 1 if device_fraud_rate[
        (row["DeviceType"], row["DeviceInfo"])
    ] > threshold else 0,
    axis=1
)                                                   # Apply binary flag over the feature 'DeviceRisk'

- Fraud rates are calculated for all device combinations.

- The top 10% highest-risk devices are automatically identified using the 90th percentile.

- The top 10% device combinations were selected because they represent the most unusually risky behavior compared to the rest of the dataset.  


A binary 'DeviceRisk' feature was engineered using 'DeviceType' and 'DeviceInfo'

- Device combinations belonging to the **top 10% highest** fraud-rate group were classified as **high-risk (1)**
- All remaining device combinations were classified as **low-risk (0)**

In [ ]:
df.drop(
    columns=["DeviceType", "DeviceInfo"],
    inplace=True
)                                               # SInce we now already have engineering feature 'DeviceRisk' we don't need the raw columns.

In [ ]:
print(df.columns)       # Verify the new dataframe structure

In [ ]:
df[[
    "AmtToMeanRatio",
    "HourOfDay",
    "DeviceRisk"
]].head()                   # Verify new features

#### We now need to define features and targets for Dataset Splitting and SMOTE

Separate x and y:

In [ ]:
x = df.drop(columns=["isFraud"])    # Define the feature set by dropping the target variable 'isFraud' from the dataset
y = df["isFraud"]                   # Define the target variable as the 'isFraud' column from the dataset

- The dataset was separated in **x** and **y**, where x defines the features in the dataset
- **y** is defined as the target column.

In [ ]:
from sklearn.model_selection import train_test_split    # Importing the train_test_split function from scikit-learn library to split the dataset into training and testing sets

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)                                                       # Split the dataset where 80% of the data is used for training and 20% is used for testing

- Performed stratified 80/20 train-test split on the dataset.



In [ ]:
print("Training Shape:", x_train.shape)     # Verify shapes for training and testing data
print("Testing Shape:", x_test.shape)

#### Scaling numerical features using **RobustScaler**

In [ ]:
from sklearn.preprocessing import RobustScaler      # Import the RobustScaler library

scaler = RobustScaler()                             # Initialize the RobustScaler for feature scaling

x_train_scaled = scaler.fit_transform(x_train)      # Apply RobustScaler
x_test_scaled = scaler.transform(x_test)

#### Apply SMOTE for Class Imbalance

In [ ]:
from imblearn.over_sampling import SMOTE                # Importing the SMOTE library from imblearn to handle class imbalance in the dataset

smote = SMOTE(random_state=42)                          # Initialize the SMOTE object with a random state for reproducibility

x_train_smote, y_train_smote = smote.fit_resample(
    x_train_scaled,
    y_train
)                                                       # Apply SMOTE to the training data to handle class imbalance by oversampling the minority class

In [ ]:
print("Before SMOTE:")

print(y_train.value_counts())                           # Display the count of each class in the original training target variable before applying SMOTE
print(y_train.value_counts(normalize=True) * 100)       # Display the percentage of each class in the original training target variable before applying SMOTE

- The original dataset is highly imbalanced

- Fraudulent transactions comprise only a small percentage of the observations

In [ ]:
print("After SMOTE:")

print(y_train_smote.value_counts())                         # Display the count of each class in the training target variable after applying SMOTE
print(y_train_smote.value_counts(normalize=True) * 100)     # Display the percentage of each class in the training target variable after applying SMOTE

- Applying SMOTE, it introduces synthetic fraud transactions samples and balance class distribution

#### Visualizing Class Distribution on Before vs After SMOTE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))

sns.countplot(
    x=y_train,
    ax=axes[0]
)                                   # Before SMOTE

axes[0].set_title("Before SMOTE")

sns.countplot(
    x=y_train_smote,
    ax=axes[1]
)                                   # After SMOTE

axes[1].set_title("After SMOTE")

plt.tight_layout()

plt.show()

- The dataset was highly imbalanced, with fraudulent transactions representing only a small percentage of observations.

- SMOTE (Synthetic Minority Oversampling Technique) was applied exclusively to the training dataset to generate synthetic fraud samples and balance class distribution.

In [ ]:
x_train = pd.DataFrame(
    x_train_smote,
    columns=x_train.columns
)                               # Convert the scaled and SMOTE-applied training features back to a DataFrame with original column names for better readability

X_test = pd.DataFrame(
    x_test_scaled,
    columns=x_test.columns
)                               # Convert the scaled test features back to a DataFrame with original column names for better readability

### **Model Training, Comparison & Threshold Optimization**

#### Import Libraries:

In [ ]:
from lightgbm import LGBMClassifier             # Importing the LightGBM library for classification
from xgboost import XGBClassifier               # Importing the XGBoost library for classification
from sklearn.ensemble import IsolationForest    # Importing the Isolation Forest algorithm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve,
    confusion_matrix,
    roc_curve,
    average_precision_score
)                                               # Importing various evaluation metrics from scikit-learn to evaluate the performance of the classification models

#### **LightGBM Model**

#### Initialize

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)                               # Initialize the LightGBM classifier with specified hyperparameters

#### Train

In [ ]:
lgbm_model.fit(
    x_train,
    y_train_smote
)                   # Fit the LightGBM model on the training data after applying SMOTE

#### **XGBoost Model**

#### Initialize

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)                               # Initialize the XGBoost classifier with specified hyperparameters

#### Train

In [ ]:
xgb_model.fit(
    x_train,
    y_train_smote
)                   # Fit the XGBoost model on the training data after applying SMOTE

#### **Isolation Forest Model**

#### Initialize

In [ ]:
iso_model = IsolationForest(
    contamination=0.035,
    random_state=42,
    n_jobs=-1
)                               # Initialize the Isolation Forest model for anomaly detection with specified hyperparameters

#### Train

In [ ]:
iso_model.fit(x_train)      # Fit the Isolation Forest model on the training data to learn the patterns of normal transactions for anomaly detection

#### Generate Predictions for all models:

LightGBM Model:

In [ ]:
lgbm_probs = lgbm_model.predict_proba(X_test)[:, 1]     # Get the predicted probabilities for the positive class (fraud) from the LightGBM model on the test set

lgbm_preds = (lgbm_probs >= 0.5).astype(int)            # Convert the predicted probabilities into binary class predictions

XGBoost Model:

In [ ]:
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]       # Get the predicted probabilities for the positive class (fraud) from the XGBoost model on the test set

xgb_preds = (xgb_probs >= 0.5).astype(int)              # Convert the predicted probabilities into binary class predictions

Isolation Forest Model:

In [ ]:
iso_preds_raw = iso_model.predict(X_test)       # Get the raw predictions from the Isolation Forest model, where -1 indicates an anomaly (potential fraud) and 1 indicates normal transactions

iso_preds = np.where(
    iso_preds_raw == -1,
    1,
    0
)                                               # Convert the raw predictions from Isolation Forest into binary class predictions

Isolation Forest outputs:

- 1 -> normal
- -1 -> anomaly

We convert:

- anomaly -> fraud (1)
- normal -> non-fraud (0)

#### Creating an Evaluation Function for all required metrics:

In [ ]:
def evaluate_model(y_true, y_pred, y_prob=None):                # Define a function to evaluate the model's performance using various metrics
    
    accuracy = accuracy_score(y_true, y_pred)                   # Calculate the accuracy of the model's predictions
    
    precision = precision_score(y_true, y_pred)                 # Calculate the precision of the model's predictions
    
    recall = recall_score(y_true, y_pred)                       # Calculate the recall of the model's predictions
    
    f1 = f1_score(y_true, y_pred)                               # Calculate the F1-score of the model's predictions
    
    roc_auc = roc_auc_score(y_true, y_pred)                     # Calculate the ROC-AUC score of the model's predictions
    
    if y_prob is not None:
        pr_auc = average_precision_score(y_true, y_prob)        # Calculate the Precision-Recall AUC score of the model's predictions if predicted probabilities are provided
    else:
        pr_auc = average_precision_score(y_true, y_pred)        # Calculate the Precision-Recall AUC score of the model's predictions using binary predictions if predicted probabilities are not provided
    
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")
    print(f"PR-AUC    : {pr_auc:.4f}")

#### Evaluating Models

- LightGBM Evaluation

In [ ]:
print("LightGBM Performance\n")

evaluate_model(
    y_test,
    lgbm_preds,
    lgbm_probs
)                                   # Evaluate the performance of the LightGBM model using the defined evaluation function

- XGBoost Evaluation

In [ ]:
print("XGBoost Performance\n")

evaluate_model(
    y_test,
    xgb_preds,
    xgb_probs
)                                    # Evaluate the performance of the XGBoost model using the defined evaluation function

- Isolation Forest Evaluation

In [ ]:
print("Isolation Forest Performance\n")

evaluate_model(
    y_test,
    iso_preds
)                                               # Evaluate the performance of the Isolation Forest model using the defined evaluation function

- Among all evaluated models, XGBoost achieved the strongest overall fraud detection performance across multiple evaluation metrics, including F1-Score, ROC-AUC, and PR-AUC.

- Isolation Forest showed significantly weaker performance compared to supervised boosting models, indicating that anomaly detection alone was insufficient for capturing complex fraud behavior.

- Based on the evaluation results, XGBoost was selected as the final model for further threshold optimization and Tuning.

#### Visualizations:

- Confusion Matrix:

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):       # Define a function to plot the confusion matrix
    
    cm = confusion_matrix(y_true, y_pred)               # Calculate the confusion matrix using the true labels and predicted labels
    
    plt.figure(figsize=(5,4))
    
    sns.heatmap(
        cm,
        annot=True,
        fmt='d'
    )
    
    plt.title(title)
    
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    
    plt.show()

#### Plotting matrices:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5)) # Create subplots

models = [
    ("LightGBM", lgbm_preds),
    ("XGBoost", xgb_preds),
    ("Isolation Forest", iso_preds)
]                                               # Define Model details

for ax, (title, preds) in zip(axes, models):
    
    cm = confusion_matrix(y_test, preds)        # Plot confusion matrices
    
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        ax=ax,
        xticklabels=["Non-Fraud", "Fraud"],
        yticklabels=["Non-Fraud", "Fraud"]
    )
    
    ax.set_title(f"{title} Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

#### Key Observations:

- The **XGBoost model** provides the **best overall** fraud detection performance among the evaluated models.
    - Correctly identified 113,162 legitimate transactions
    - Correctly detected 2,055 fraudulent transactions
    - Missed 2,078 fraud cases 
    - Incorrectly flagged 813 legitimate transactions as fraud

- it achieved:

    - **the highest fraud detection capability,**
    - **the lowest false positive rate,**
    - **and the best balance between precision and recall.**

- The **LightGBM model** demonstrates strong capability in identifying legitimate transactions while maintaining a balanced fraud detection rate.
    - Correctly identified 113,065 legitimate transactions
    - Correctly detected 1,881 fraudulent transactions
    - Missed 2,252 fraud cases (False Negatives)
    - Incorrectly flagged 910 legitimate transactions as fraud (False Positives)

- **The model achieves a good balance between fraud detection.**

- **Isolation Forest** struggled to accurately identify fraudulent transactions compared to the supervised learning models.
    - Correctly identified 112,458 legitimate transactions
    - Correctly detected only 245 fraudulent transactions
    - Missed 3,888 fraud cases
    - Incorrectly flagged 1,517 legitimate transactions as fraud

- The model achieved:
    - **very low fraud detection capability,**
    - **high missed fraud volume,**

- Since Isolation Forest is an unsupervised anomaly detection model, it lacks the ability to learn fraud patterns as effectively as supervised models like LightGBM and XGBoost.

#### ROC Curve

- ROC Curve shows True Positive Rate vs False Positive Rate

In [ ]:
iso_probs = -iso_model.decision_function(X_test)        # ensure iso_probs contains anomaly scores converted appropriately for isolation forest model

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models = [
    ("LightGBM", lgbm_probs),
    ("XGBoost", xgb_probs),
    ("Isolation Forest", iso_probs)
]

for ax, (model_name, probs) in zip(axes, models):       # Plot ROC curves
    
    fpr, tpr, _ = roc_curve(y_test, probs)              # Calculate ROC metrics
    
    auc_score = roc_auc_score(y_test, probs)            # Calculate ROC-AUC score
    
    ax.plot(
        fpr,
        tpr,
        label=f"AUC = {auc_score:.4f}"
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle='--'
    )
    
    ax.set_title(f"{model_name} ROC Curve")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    

plt.tight_layout()

plt.show()

ROC Curves were plotted to evaluate the tradeoff between:
- True Positive Rate (Recall),
- and False Positive Rate

across different classification thresholds.

ROC-AUC was used to measure the overall separability between fraudulent and non-fraudulent transactions. Higher ROC-AUC values indicate stronger classification capability.

- XGBoost:

    - **best separability**
    - **strongest ability to distinguish fraud vs non-fraud**

- LightGBM:

    - also strong
    - **slightly weaker separability than XGBoost**
    - still highly effective

- Isolation Forest:

    - **significantly weaker**
    - although better than random guessing

#### Precision-Recall Curve

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = [
    ("LightGBM", lgbm_probs),
    ("XGBoost", xgb_probs),
    ("Isolation Forest", iso_probs)
]                                                       # Define model details for plotting Precision-Recall curves

for ax, (model_name, probs) in zip(axes, models):
    
    precision, recall, _ = precision_recall_curve(      # Calculate precision-recall values
        y_test,
        probs
    )
    
    pr_auc = average_precision_score(                   # Calculate PR-AUC
        y_test,
        probs
    )
    
    ax.plot(
        recall,
        precision,
        label=f"PR-AUC = {pr_auc:.4f}"
    )
    
    ax.set_title(f"{model_name} PR Curve")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    
    ax.legend()

plt.tight_layout()

plt.show()

- Precision-Recall Curves were plotted to evaluate model performance on the minority fraud class across different classification thresholds.

#### Threshold Optimization

- Optimize threshold using F1-Score

- First, compute F1-Score across threshold:

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.05)              # Define a range of thresholds to evaluate the F1-score at different classification thresholds for the XGBoost model

f1_scores = []                                      # Initialize an empty list to store F1-scores for each threshold

for threshold in thresholds:
    
    preds = (xgb_probs >= threshold).astype(int)    # Convert predicted probabilities into binary class predictions based on the current threshold
    
    score = f1_score(y_test, preds)                 # Calculate the F1-score for the current threshold   
    f1_scores.append(score)                         # Append the calculated F1-score to the list of F1-scores for each threshold

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    thresholds,
    f1_scores,
    marker='o'
)                                               # Plot the F1-score against the thresholds to visualize how the F1-score changes with different classification thresholds for the XGBoost model

plt.title("XGBoost Threshold vs F1-Score")

plt.xlabel("Threshold")
plt.ylabel("F1-Score")


plt.show()

In [ ]:
best_threshold = thresholds[
    np.argmax(f1_scores)
]                                               # Identify the threshold that gives the highest F1-score for the XGBoost model

print("Best Threshold:", best_threshold)

In [ ]:
optimized_preds = (
    xgb_probs >= best_threshold
).astype(int)                       # Get the optimized predictions based on the best threshold identified

In [ ]:
print("Optimized XGBoost Performance\n")

evaluate_model(
    y_test,
    optimized_preds,
    xgb_probs
)                                               # Evaluate the performance of the XGBoost model using the optimized predictions based on the best threshold identified

- This indicates that the default classification boundary was already near-optimal

- The model already learned a good probability separation

- The optimal threshold was observed near the default value, resulting in performance metrics that remained unchanged.

#### Hyperparameter Tuning using Optuna

In [ ]:
import optuna                                       # Importing the Optuna library for hyperparameter optimization

def objective(trial):                               # Define Optuna optimization objective function
    
    params = {                                      # Define hyperparameter search space
        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            400
        ),                                          # Number of boosting trees
        
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2
        ),                                          # Learning rate controlling contribution of each tree
        
        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),                                          # Maximum depth of each decision tree
        
        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),                                          # Fraction of training samples used per tree
        
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),                                          # Fraction of features used per tree
        
        "random_state": 42,
        "n_jobs": -1,
        "eval_metric": "logloss"
    }
    
    model = XGBClassifier(**params)                 # Initialize XGBoost model with trial parameters
    
    model.fit(
        x_train,
        y_train_smote                               # Train model
    )
    
    probs = model.predict_proba(
        X_test                                      # Generate fraud probabilities for test data
    )[:, 1]
    
    score = average_precision_score(
        y_test,
        probs                                       # Compute PR-AUC score
    )
    
    return score

This defines the optimization function that Optuna repeatedly executes.

Each trial:
- Selects a new hyperparameter combination,
- Trains a model,
- Evaluates performance,
- Returns a score.

In [ ]:
study = optuna.create_study(
    direction="maximize"
)                               # Create an Optuna study object to maximize the objective function (PR-AUC score)

# Run optimization
study.optimize(
    objective,
    n_trials=20
)                               # Run the optimization for 20 trials to find the best hyperparameters for the XGBoost model based on PR-AUC score

In [ ]:
print("Best Parameters:\n")

print(study.best_params)        # Display the best hyperparameters found by Optuna for the XGBoost model based on PR-AUC score

In [ ]:
best_model = XGBClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)                               # Initialize the final XGBoost model with the best hyperparameters found by Optuna

best_model.fit(
    x_train,
    y_train_smote
)                               # Fit the final XGBoost model using the best hyperparameters found by Optuna

In [ ]:
best_probs = best_model.predict_proba(
    X_test
)[:, 1]                                     # Generate predicted probabilities for the test set using the final XGBoost model with optimized hyperparameters

best_preds = (
    best_probs >= best_threshold
).astype(int)                               # Convert predicted probabilities to binary predictions based on the optimal threshold

In [ ]:
print("Final Tuned XGBoost Performance\n")

evaluate_model(
    y_test,
    best_preds,
    best_probs
)                        # Evaluate the performance of the final tuned XGBoost model using the optimized predictions based on the best threshold identified 

These are a major improvement over your earlier XGBoost results.

| | Before Tuning | After Tuning |
|---|---|---|
| Accuracy | 0.9755 | 0.9859 |
| Precision | 0.7165 | 0.9240 |
| Recall | 0.4972 | 0.6504 |
| F1-Score | 0.5871 | 0.7634 |
| ROC-AUC | 0.7450 | 0.8242 |
| PR-AUC | 0.6081 | 0.8246 |


Hyperparameter optimization using Optuna significantly improved fraud detection performance across all major evaluation metrics.

The tuned XGBoost model achieved:
- substantially higher Precision,
- improved Recall,
- stronger F1-Score,
- and significantly better PR-AUC performance.

The results indicate that the optimized model learned more effective fraud decision boundaries and generated higher-quality fraud probability estimates.
- **Fewer false fraud alerts**
- **Results in reduced investigation cost**
- **Fewer legitimate customers incorrectly flagged.**

**XGBoost achieved the best overall performance for fraud detection.**

Reasons:
- Strong capability to learn complex fraud patterns
- Excellent handling of high-dimensional tabular data
- Better precision-recall tradeoff
- Highest PR-AUC among all evaluated models

### **Explainable AI with SHAP Values**

What is Explainable AI with SHAP?

- Explainable AI (XAI) using SHAP (SHapley Additive exPlanations) is a framework used to interpret machine learning models by assigning an importance value to each feature for a specific prediction. It answers "Why did the AI make this decision?" by explaining how much each data attribute pushed the output away from an average baseline.

- SHAP computes the contribution of a feature by calculating the difference in the model's prediction with and without that feature across all possible combinations of features.

#### Import SHAP Library

In [ ]:
import shap         # Importing the SHAP library for model interpretability and explainability

#### Initialize SHAP Explainer

Since XGBoost is our best model, we will use 'TreeExplainer', which is optimized for tree based models.

In [ ]:
explainer = shap.TreeExplainer(best_model)      # Create a SHAP explainer object for the final XGBoost model to compute SHAP values for feature importance and interpretability analysis

#### Generate SHAP Values

In [ ]:
shap_values = explainer.shap_values(X_test)     # Calculate SHAP values for the test set using the SHAP explainer object to understand the contribution of each feature to the model's predictions

#### Convert Test Data Back to DataFrame

SHAP plots become easier to read with feature names.

In [ ]:
X_test_df = pd.DataFrame(
    X_test,
    columns=x.columns
)                                   # Convert the test features back to a DataFrame with original column names for better readability in SHAP plots

Global SHAP Summary Plot

This shows:

- most important features overall
- feature impact direction
- feature importance magnitude

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_df,
    max_display=20
)                               # Create a SHAP summary plot to visualize the impact of the top 20 features on the model's predictions for the test set

In [ ]:
prediction_probs = best_model.predict_proba(
    X_test
)[:,1]                                          # Get the predicted probabilities for the positive class (fraud) from the final XGBoost model on the test set for SHAP dependence plot

### SHAP Waterfall Plots:

#### Confirmed Fraud Case

High fraud probability.

In [ ]:
fraud_index = np.argmax(prediction_probs)            # Identify the index of the transaction with the highest predicted probability of being fraud

print("Fraud Probability:",
      prediction_probs[fraud_index])                 # Display the predicted probability of fraud for the transaction with the highest predicted probability of being fraud

In [ ]:
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[fraud_index],
        base_values=explainer.expected_value,
        data=X_test_df.iloc[fraud_index],
        feature_names=x.columns.tolist()
    )
)                                                   # Create a SHAP waterfall plot to visualize the contribution of each feature to the prediction for Confirmed Fraud Transaction

- This plot explains the feature-level reasoning behind a high fraud prediction.

- Positive SHAP values push the prediction toward fraud.

- Negative SHAP values reduce fraud probability.

#### Borderline Case

Closest probability to 0.50.

In [ ]:
borderline_index = np.argmin(
    np.abs(prediction_probs - 0.50)             # Identify the index of the transaction with the predicted probability closest to 0.5, which represents a borderline case between fraud and non-fraud
)

print("Borderline Probability:",
      prediction_probs[borderline_index])       # Display the predicted probability of fraud for the transaction for the borderline case between fraud and non-fraud

In [ ]:
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[borderline_index],
        base_values=explainer.expected_value,
        data=X_test_df.iloc[borderline_index],
        feature_names=x.columns.tolist()
    )
)                                    # Create a SHAP waterfall plot to visualize the contribution of each feature to the prediction for the Borderline Case between Fraud and Non-Fraud Transaction

- This transaction lies near the decision boundary.

- The model identified both fraud-indicating and legitimate signals, resulting in an uncertain prediction close to 50% probability.

#### Legitimate Transaction

Lowest fraud probability.

In [ ]:
legit_index = np.argmin(prediction_probs)       # Identify the index of the transaction with the lowest predicted probability of being fraud, which represents a legitimate transaction

print("Legitimate Probability:",
      prediction_probs[legit_index])            # Display the predicted probability of fraud for the transaction for legitimate transaction

In [ ]:
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[legit_index],
        base_values=explainer.expected_value,
        data=X_test_df.iloc[legit_index],
        feature_names=x.columns.tolist()
    )
)                                           # Create a SHAP waterfall plot to visualize the contribution of each feature to the prediction for the Legitimate Transaction

- This transaction was classified as legitimate with very low fraud probability.

- Most feature contributions reduced the likelihood of fraud.

#### SHAP Dependence Plot

Dependence plots show:

- How a feature affects fraud prediction interaction behavior
- Uses the most important SHAP feature.

Find Most Important Feature: 

In [ ]:
shap_importance = np.abs(shap_values).mean(axis=0)      # Calculate the mean absolute SHAP value for each feature across all samples in the test set to determine feature importance

top_feature = x.columns[
    np.argmax(shap_importance)
]                                                       # Identify the feature with the highest mean absolute SHAP value, which indicates the most important feature contributing to the model's predictions

print("Top Feature:", top_feature)                      # Display the name of the most important feature based on mean absolute SHAP values

#### Generate Dependence Plot

In [ ]:
shap.dependence_plot(
    top_feature,
    shap_values,
    X_test_df
)                           # Create a SHAP dependence plot to visualize the relationship between the most important feature and the model's predictions

#### Observations:

- The relationship is highly nonlinear.

- Fraud likelihood changes depending on transaction timing.

- The model learned that certain hours are more suspicious.

- Positive SHAP values indicate increased fraud likelihood, while negative values reduce fraud probability.

- The curve is not smooth, instead it forms vertical bands because 'HourOfDay' is discrete, and scaling transformed them into grouped numeric levels.

#### **Confirmed Fraud Case**

The transaction was classified as fraudulent because several high-risk signals were simultaneously detected.

- unusually high transaction amount
- suspicious device characteristics
- abnormal transaction timing
- elevated engineered risk features

These factors collectively pushed the fraud probability significantly higher.

#### **Borderline Transaction**

The transaction displayed a mix of suspicious and legitimate characteristics.

Some features increased fraud probability, while others reduced it.

As a result, the model remained uncertain and assigned a probability close to 50%.

#### **Legitimate Transaction**

The transaction appeared consistent with normal customer behavior.

- standard transaction amount
- low-risk device profile
- typical behavioral patterns

These signals collectively reduced the fraud probability.

#### SHAP Importance vs XGBoost Feature Importance:

XGBoost Feature Importance

In [ ]:
xgb_importance = pd.DataFrame({
    "Feature": x.columns,
    "XGBoost Importance": best_model.feature_importances_   # Create a DataFrame to display the feature importance scores from the final XGBoost model based on the best hyperparameters found by Optuna
})                                      

xgb_importance = xgb_importance.sort_values(
    by="XGBoost Importance",
    ascending=False                                         # Sort the DataFrame by feature importance scores in descending order to identify the most important features according to the XGBoost model
)

SHAP Importance

In [ ]:
shap_importance_df = pd.DataFrame({
    "Feature": x.columns,
    "SHAP Importance": np.abs(shap_values).mean(axis=0)     # Create a DataFrame to display the mean absolute SHAP values for each feature to determine feature importance based on SHAP values
})

shap_importance_df = shap_importance_df.sort_values(
    by="SHAP Importance",
    ascending=False                                         # Sort the DataFrame by SHAP importance scores in descending order to identify the most important features based on SHAP values
)

In [ ]:
importance_comparison = pd.merge(
    xgb_importance.head(20),
    shap_importance_df.head(20),           # Merge the top 20 features from both XGBoost feature importance and SHAP importance into a single DataFrame for comparison
    on="Feature",
    how="inner"
)

importance_comparison

In [ ]:
top_importance_features = importance_comparison.head()    # Select the top features from the merged importance comparison DataFrame for visualization

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=top_importance_features,
    x="XGBoost Importance",                               # Create a bar plot to visualize the feature importance scores from the XGBoost model for the top features
    y="Feature",
    ax=axes[0]
)

axes[0].set_title("XGBoost Feature Importance")

sns.barplot(
    data=top_importance_features,
    x="SHAP Importance",
    y="Feature",                                          # Create a bar plot to visualize the mean absolute SHAP values for the top features to understand their importance based on SHAP values
    ax=axes[1]
)

axes[1].set_title("SHAP Feature Importance")

plt.tight_layout()

plt.show()

- The comparison between SHAP importance and native XGBoost feature importance revealed notable differences in feature influence rankings.

- While XGBoost importance reflects how frequently features contribute to decision tree construction, SHAP importance measures the actual impact of features on fraud predictions.

For example:
- some features exhibited high structural importance during model training,
- while others demonstrated stronger influence on final fraud classification decisions.

This comparison highlights the importance of explainability analysis beyond traditional feature importance metrics and provides deeper insight into the behavioral patterns learned by the fraud detection model.

### **Risk Segmentation & Fraud Pattern Analysis**

We will use the tuned XGBoost model to estimate fraud probability for each transaction

In [ ]:
risk_df = X_test.copy()                         # Create a new DataFrame to analyze the risk of transactions based on the predicted probabilities from the final XGBoost model

risk_df["FraudProbability"] = best_probs        # Add a new column to the risk DataFrame to store the predicted probabilities of fraud for each transaction from the final XGBoost model

risk_df["ActualFraud"] = y_test.values          # Add a new column to the risk DataFrame to store the actual fraud labels for each transaction

In [ ]:
risk_df["RiskTier"] = np.where(
    risk_df["FraudProbability"] >= 0.75,        # Classify transactions with a predicted fraud probability of 0.75 or higher as "Critical Risk"
    "Critical Risk",
    
    np.where(
        risk_df["FraudProbability"] >= 0.40,    # Classify transactions with a predicted fraud probability between 0.40 and 0.75 as "Suspicious" or "Clear"
        "Suspicious",
        "Clear"
    )
)

In [ ]:
risk_df[[
    "FraudProbability",
    "RiskTier"
]].head()                   # Verify the new segmentation of transactions into risk tiers

- Transactions were segmented using predicted fraud probabilities generated by the tuned XGBoost model.

Risk tiers were defined as:

- Critical Risk → fraud probability ≥ 0.75
- Suspicious → fraud probability between 0.40 and 0.74
- Clear → fraud probability < 0.40

This segmentation approach helps prioritize fraud investigation efforts based on transaction risk severity.

In [ ]:
tier_counts = risk_df["RiskTier"].value_counts()    # Calculate the count of transactions in each risk tier to understand the distribution of transactions across different risk levels

print(tier_counts)

In [ ]:
plt.figure(figsize=(7,5))

sns.countplot(
    data=risk_df,
    x="RiskTier",
    order=["Critical Risk", "Suspicious", "Clear"]      # Create a count plot to visualize the distribution of transactions across the defined risk tiers (Critical Risk, Suspicious, Clear)
)

plt.title("Transaction Distribution by Risk Tier")

plt.xlabel("Risk Tier")
plt.ylabel("Transaction Count")

plt.show()

- Most transactions were classified as Clear, indicating low predicted fraud probability.

- A smaller subset was categorized as Suspicious or Critical Risk, allowing fraud analysts to focus investigative resources on the highest-risk transactions.

#### Compute Average Transaction Amount

In [ ]:
risk_df["TransactionAmt"] = x_test["TransactionAmt"].values     # Add the original transaction amounts from the test set because scaled data can cause issues in interpretation of the average transaction amount

avg_transaction_amt = risk_df.groupby(
    risk_df["RiskTier"]                                         # Calculate the average transaction amount for each risk tier
)["TransactionAmt"].mean()

print(avg_transaction_amt)

In [ ]:
avg_transaction_amt.plot(                               # Create a bar plot to visualize the average transaction amount for each risk tier to understand how transaction amounts vary across different risk levels
    kind='bar',
    figsize=(7,5)
)

plt.title("Average Transaction Amount by Risk Tier")

plt.xlabel("Risk Tier")
plt.ylabel("Average Transaction Amount")

plt.xticks(rotation=0)

plt.show()

- Suspicious transactions showed the highest average transaction amount, while Critical Risk transactions exhibited moderately elevated values.

#### Device Type Distribution

- This helps identify whether specific device categories are associated with fraud risk

In [ ]:
device_df = df_identity[[                                           # Create a new DataFrame to merge the device information
    "TransactionID",
    "DeviceType"
]]


risk_df["TransactionID"]=df_identity["TransactionID"].astype(int)   # Recover the 'TransactionID' column to merge the device information


risk_df = risk_df.merge(
    device_df,
    on="TransactionID",                                             # Merge the device information with the risk DataFrame to analyze the distribution of device types across different risk tiers
    how="left"
)

In [ ]:
device_distribution = pd.crosstab(          # Analyze the distribution of device types across different risk tiers using a crosstab
    risk_df["RiskTier"],
    risk_df["DeviceType"]
)

device_distribution

In [ ]:
device_distribution_pct = pd.crosstab(      # Create a crosstab to show the distribution of device types across different risk tiers
    risk_df["RiskTier"],
    risk_df["DeviceType"],
    normalize="index"
) * 100

device_distribution_pct

In [ ]:
device_distribution_pct.plot(                                   # Create a bar plot to visualize the percentage distribution of device types across different risk tiers
    kind='bar',
    figsize=(10,6)
)

plt.title("Device Type Percentage Distribution by Risk Tier")

plt.xlabel("Risk Tier")
plt.ylabel("Percentage")

plt.legend(title="DeviceType")

plt.show()

- Desktop devices represented the majority of transactions across all tiers.
- The proportional distribution of device types remained relatively consistent between Clear, Suspicious, and Critical Risk categories.

#### Hour-of-day Pattern

In [ ]:
risk_df["HourOfDay"] = x_test["HourOfDay"].values       # Recover the pre-scaled 'HourOfDay' feature from the test set because scaled data can cause issues in interpretation of the hour distribution of transactions

hour_pattern = risk_df.groupby(                         # Computing hour distribution of transactions
    ["HourOfDay", "RiskTier"]
).size().reset_index(name="Count")

In [ ]:
plt.figure(figsize=(12,6))                                  # Creating a line plot to visualize the hour-of-day transaction pattern for each risk tier

sns.lineplot(
    data=hour_pattern,
    x="HourOfDay",
    y="Count",
    hue="RiskTier"
)

plt.title("Hour-of-Day Transaction Pattern by Risk Tier")

plt.xlabel("Hour of Day")
plt.ylabel("Transaction Count")

plt.show()

- Critical Risk transactions showed relatively stronger concentration during specific periods of the day.

- Clear transactions followed broader overall transaction volume patterns.

- Transaction timing contributed meaningfully to fraud prediction and behavioral segmentation.

#### Grouped Bar Chart Comparing Risk Tiers

We combine:

- average amount
- average fraud probability

In [ ]:
tier_summary = risk_df.groupby("RiskTier").agg({        # Calculate summary statistics for each risk tier including average transaction amount and average fraud probability
    "TransactionAmt": "mean",
    "FraudProbability": "mean"
}).reset_index()


tier_summary_normalized = tier_summary.copy()           # Create a copy of the tier summary DataFrame to normalize the values for better comparison across risk tiers

tier_summary_normalized["TransactionAmt"] = (
    tier_summary_normalized["TransactionAmt"] /
    tier_summary_normalized["TransactionAmt"].max()     # Normalize the average transaction amount by dividing by the maximum average transaction amount across all risk tiers to scale it between 0 and 1 for better comparison
)

tier_summary_normalized["FraudProbability"] = (
    tier_summary_normalized["FraudProbability"] /
    tier_summary_normalized["FraudProbability"].max()   # Normalize the average fraud probability by dividing by the maximum average fraud probability across all risk tiers to scale it between 0 and 1 for better comparison
)

tier_summary_normalized

In [ ]:
tier_summary_normalized.set_index(              # Create a bar plot to compare the normalized average transaction amount and normalized average fraud probability across different risk tiers
    "RiskTier"
)[[
    "TransactionAmt",
    "FraudProbability"
]].plot(
    kind='bar',
    figsize=(10,6)
)

plt.title("Normalized Risk Tier Comparison")

plt.xlabel("Risk Tier")
plt.ylabel("Normalized Value")

plt.xticks(rotation=0)

plt.show()

A normalized grouped bar chart was created to compare:

- average transaction amount
- average fraud probability


across all risk tiers.

- Normalization was applied to ensure comparability between metrics operating on different numerical scales.

- Critical Risk transactions exhibited the highest fraud probability.

- Suspicious transactions showed the highest normalized transaction amounts.

- Clear transactions maintained very low fraud probability despite moderate transaction values.

### Top 3 Fraud Patterns

- Extract Critical Risk Transactions

In [ ]:
critical_risk_df = risk_df[
    risk_df["RiskTier"] == "Critical Risk"      # Filter the risk DataFrame to analyze only the transactions classified as "Critical Risk"
]

#### High Transaction Amount

In [ ]:
critical_risk_df["TransactionAmt"].describe()   # Display the descriptive statistics for the transaction amounts

#### Device Risk Distribution

In [ ]:
critical_risk_df["DeviceRisk"].value_counts()       # Analyze the distribution of the engineerred features

#### High-Risk Hours

In [ ]:
critical_risk_df["HourOfDay"].value_counts().head(10)   # Analyze the distribution of transactions across different hours of the day

#### 1. High Transaction Amounts
Critical Risk transactions frequently involved unusually large transaction values.

This indicates that abnormal spending behavior is a strong fraud indicator.


#### 2. Risky Device Characteristics
High-risk transactions were more commonly associated with suspicious device profiles.

This confirms that device-related behavioral signals contribute significantly to fraud detection.


#### 3. Unusual Transaction Timing
Fraudulent transactions appeared more frequently during specific hours of the day.

This suggests that attackers may exploit lower monitoring periods to conduct fraudulent activity.

### Actual Fraud Rate Per Tier:

- Fraud Rate by Tier:

In [ ]:
fraud_rate_by_tier = risk_df.groupby(
    "RiskTier"
)["ActualFraud"].mean() * 100               # Calculate the fraud rate for each risk tier by grouping the risk DataFrame by "RiskTier"

print(fraud_rate_by_tier)

In [ ]:
fraud_rate_by_tier.plot(                        # Create a bar plot to visualize the actual fraud rate for each risk tier
    kind='bar',
    figsize=(7,5)
)

plt.title("Actual Fraud Rate by Risk Tier")

plt.xlabel("Risk Tier")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=0)

plt.show()

- Critical Risk transactions exhibited the highest actual fraud rate.

- This validates the effectiveness of the probability-based segmentation framework and demonstrates that the model successfully prioritized high-risk transactions.

### **Streamlit Fraud Operations Dashboard**

In [ ]:
import joblib                   # Importing the joblib library for saving and loading the trained model
import plotly.express as px     # Importing the Plotly Express library for interactive visualizations
import streamlit as st          # Importing the Streamlit library for building interactive web applications

In [ ]:
joblib.dump(
    best_model,
    "dashboard/model.pkl"       # Save the final tuned XGBoost model with optimized hyperparameters to a file named "model.pkl"
)


risk_df.to_csv(
    "dashboard/risk_df.csv",    # Save the risk DataFrame with transaction details and risk tiers to a CSV file named "risk_df.csv" for use in the Streamlit dashboard
    index=False
)

A multi-page Streamlit dashboard was developed to operationalize the fraud detection system.

Key capabilities:
- real-time fraud monitoring
- transaction exploration
- interactive visual analytics
- SHAP-based explainability
- risk segmentation analysis

The deployment code is available in the accompanying Kaggle dataset:

streamlit_app.zip

The dashboard contains:
- Fraud monitoring KPIs
- Transaction exploration
- SHAP explainability
- Risk segmentation

### **Visualizations**

#### SHAP Global Summary Plot

In [ ]:
plt.figure(figsize=(12,8))

shap.summary_plot(          # Create a SHAP summary plot to visualize the impact of the top features on the model's predictions
    shap_values,
    X_test_df,
    max_display=20
)

- The SHAP summary plot visualizes the overall contribution of features toward fraud predictions.

- Features at the top of the chart have the greatest influence on model behavior.

#### Fraud Rate by Hour of Day

In [ ]:
hourly_fraud_rate = risk_df.groupby(            # Analyze the hourly fraud rate by grouping the risk DataFrame by "HourOfDay" and calculating the mean of the "ActualFraud" column
    "HourOfDay"
)["ActualFraud"].mean() * 100

hourly_fraud_rate = hourly_fraud_rate.reset_index() # Reset the index of the hourly fraud rate DataFrame for better visualization

In [ ]:
plt.figure(figsize=(12,6))

sns.lineplot(
    data=hourly_fraud_rate,
    x="HourOfDay",
    y="ActualFraud",
    marker="o"
)

plt.title("Fraud Rate by Hour of Day")

plt.xlabel("Hour of Day")
plt.ylabel("Fraud Rate (%)")

plt.show()

- Fraud activity varied across different hours of the day.

- Certain time periods exhibited elevated fraud rates, indicating potential behavioral fraud patterns associated with transaction timing.

#### TransactionAmt distribution

Compare 'fraud' and 'non-fraud' transactions

Using log scale because fraud datasets are skewed.

In [ ]:
plt.figure(figsize=(12,6))  # Create a histogram to visualize the distribution of transaction amounts for fraudulent and non-fraudulent transactions on a log scale

sns.histplot(
    data=risk_df,
    x="TransactionAmt",
    hue="ActualFraud",
    bins=100,
    log_scale=True,
    element="step"
)

plt.title("Transaction Amount Distribution")

plt.xlabel("Transaction Amount (Log Scale)")
plt.ylabel("Frequency")

plt.show()

- Transaction amounts displayed strong skewness.

#### Risk tier donut chart

In [ ]:
risk_counts = risk_df["RiskTier"].value_counts()    # Calculate the count of transactions in each risk tier to visualize the distribution of transactions across different risk levels

In [ ]:
plt.figure(figsize=(7,7))

plt.pie(
    risk_counts.values,
    labels=risk_counts.index,
    autopct='%1.2f%%',
    wedgeprops={"width": 0.6}
)

plt.title("Risk Tier Distribution")

plt.show()

The majority of transactions were classified as **Clear** transactions, accounting for approximately **97.33%** of the dataset.

A smaller proportion of transactions were identified as:
- **Critical Risk** transactions (~1.99%)
- **Suspicious** transactions (~0.68%)

#### Precision-Recall curve with optimal threshold

Compute PR Curve;

In [ ]:
precision, recall, thresholds = precision_recall_curve(
    y_test,
    best_probs
)                       # Calculate precision-recall values for the final tuned XGBoost model

In [ ]:
f1_scores = (
    2 * precision * recall
) / (precision + recall + 1e-10)

best_index = np.argmax(f1_scores)

optimal_threshold = thresholds[
    best_index
]

print("Optimal Threshold:",
      optimal_threshold)            # Identify and display the optimal classification threshold based on the highest F1-score for the final tuned XGBoost model

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(
    recall,
    precision,
    label="PR Curve"
)

plt.scatter(
    recall[best_index],
    precision[best_index],
    label=f"Optimal Threshold = {optimal_threshold:.2f}"
)

plt.title(
    "Precision-Recall Curve with Optimal Threshold"
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.legend()

plt.show()

The analysis identified an optimal threshold of approximately **0.31**, which provided the best balance between Precision and Recall based on F1-Score optimization.

At this threshold:
- The model maintained high fraud prediction precision,
- Significantly improves fraud detection coverage.

#### Interactive Plotly Scatter Plot Colored by fraud probability

In [ ]:
fig = px.scatter(
    risk_df.sample(10000),   # sample for readability
    
    x="HourOfDay",          # Plotting the relationship between the hour of day and transaction amount colored by fraud probability
    y="TransactionAmt",      
    
    color="FraudProbability",
    
    title="Transaction Amount vs Hour of Day",
    
    color_continuous_scale="Turbo",
    
    opacity=0.6
)

fig.update_traces(          # Customize the appearance of the scatter plot points
    marker=dict(size=4)
)

fig.update_layout(
    xaxis_title="Hour of Day",
    yaxis_title="Transaction Amount",
    title_x=0.5
)

fig.update_yaxes(type="log")

fig.show(renderer="browser")

An interactive scatter plot was created to analyze the relationship between:
- transaction amount,
- transaction timing,
- and predicted fraud probability.

The visualization revealed that:
- most transactions exhibited low fraud probability,
- while higher-risk transactions appeared across multiple transaction hours and amount ranges.

- Applying logarithmic scaling to transaction amounts improved visibility of both small and high-value transactions, allowing clearer identification of potential fraud outliers and suspicious behavioral patterns.


`The visualization was rendered in the browser instead of directly inside the notebook due to Plotly renderer compatibility issues within the current Jupyter/VS Code environment. Rendering through the browser ensured stable interactive visualization support without affecting analytical results.`

### **Insights & Business Recommendations**

#### Which model performed best and why?

XGBoost achieved the strongest overall fraud detection performance among all evaluated models.

Key reasons for superior performance:

- Higher PR-AUC compared to LightGBM and Isolation Forest
- Better ability to learn complex fraud behavior patterns
- Strong handling of high-dimensional tabular transaction data
- Improved balance between precision and recall
- Better fraud detection capability after Optuna hyperparameter tuning

Threshold optimization further improved the model's effectiveness by selecting a probability cutoff more appropriate for imbalanced fraud classification.

Overall, XGBoost demonstrated the most reliable performance for identifying fraudulent transactions while minimizing false negatives.

####  Why PR-AUC matters more than accuracy in fraud detection?

The dataset is highly imbalanced, with fraudulent transactions representing only a small percentage of total observations.

In such cases, accuracy becomes misleading.

For example:
- A model predicting every transaction as non-fraudulent could still achieve very high accuracy while completely failing to detect fraud.

Precision-Recall AUC (PR-AUC) is more appropriate because it focuses specifically on fraud detection performance.

PR-AUC evaluates:
- how effectively fraudulent transactions are identified (recall)
- how reliable fraud predictions are (precision)

This makes PR-AUC a more meaningful metric for real-world fraud detection systems where identifying minority fraud cases is critical.

#### Top 3 fraud signals identified by SHAP

SHAP analysis identified several features that strongly influenced fraud predictions.

1. Transaction Amount
Unusually large transaction amounts significantly increased fraud probability.

Fraudulent transactions frequently displayed abnormal spending behavior compared to legitimate transactions.


2. Device Risk Characteristics
Transactions associated with high-risk device profiles contributed strongly toward fraud classification.

Device-related behavioral patterns emerged as important fraud indicators.


3. Transaction Timing
Transactions occurring during unusual hours demonstrated elevated fraud probability.

This suggests that attackers may exploit lower monitoring periods to conduct suspicious activity.

#### Common characteristics of Critical Risk transactions

Critical Risk transactions displayed several recurring behavioral patterns:

- Higher-than-average transaction amounts
- Increased association with risky device profiles
- Elevated fraud probabilities generated by the XGBoost model
- Concentration during unusual transaction hours
- Strong positive SHAP contributions from engineered risk features

These characteristics indicate that fraud behavior differs significantly from legitimate customer activity and can be effectively captured using machine learning models.

#### 2 actionable fraud prevention policies

1. Real-Time Transaction Blocking for Critical Risk Transactions
Transactions classified as Critical Risk should trigger:
- real-time alerts
- manual review workflows
- temporary transaction holds

This can significantly reduce financial fraud exposure.


2. Adaptive Multi-Factor Authentication (MFA)
Additional authentication should be triggered for:
- unusually large transactions
- risky device profiles
- suspicious transaction timing patterns

Adaptive verification reduces fraud risk while minimizing customer friction for low-risk users.

#### Estimated money saved annually

In [ ]:
fraud_transactions = risk_df[                   # Filter the risk DataFrame to analyze only the transactions that were actually fraudulent based on the "ActualFraud" column
    risk_df["ActualFraud"] == 1
]

average_fraud_amount = fraud_transactions[      # Calculate the average transaction amount for the fraudulent transactions to estimate potential savings from fraud detection
    "TransactionAmt"
].mean()

detected_fraud_count = len(fraud_transactions)  # Calculate the count of detected fraudulent transactions to estimate potential savings from fraud detection

estimated_savings = (
    average_fraud_amount *                      # Estimate the potential annual savings by multiplying the average fraud amount by the count of detected fraudulent transactions
    detected_fraud_count
)

print("Estimated Annual Savings:",
      round(estimated_savings, 2))

Based on the detected fraudulent transactions and their average transaction amounts, the fraud detection system has the potential to prevent substantial financial losses annually.

Estimated savings were calculated using:
- total detected fraud cases
- average fraudulent transaction amount

Although simplified, this estimate demonstrates the potential business value of deploying automated fraud monitoring systems.

#### Model limitations

Despite strong performance, several limitations remain:

- Fraud patterns evolve over time and may reduce long-term model effectiveness.

- Some transactions may still generate false positives, impacting customer experience.

- The model was trained on historical data and may not fully generalize to future fraud behavior.

- Missing or incomplete identity information may reduce predictive capability.

#### Additional data that could improve performance


Model performance could improve further with access to additional behavioral and contextual information, including:

- Customer transaction history

- Geolocation and IP address intelligence

- Merchant reputation data

- Historical chargeback records

- Device fingerprinting information

- Network-level fraud relationships

- Real-time behavioral tracking data

Additional contextual features would help the model identify more sophisticated fraud patterns.

---

The fraud detection system successfully combined machine learning, explainable AI, and risk segmentation to identify high-risk financial transactions.

Key achievements:
- Strong fraud detection performance using XGBoost
- Improved interpretability through SHAP explainability
- Effective probability-based risk segmentation
- Interactive fraud monitoring dashboard using Streamlit

The project demonstrates how machine learning can support operational fraud prevention workflows while improving decision-making transparency and financial risk management.